# Lab 8: Discrete Derivatives as Convolutional Filters

**Computer Vision Course**

Building on Lab 7 where you learned convolution, today you'll discover one of its most powerful applications: **edge detection**.

**What you'll learn:**
- How derivatives detect edges
- Sobel filters for directional gradients
- Laplacian filter for edge detection
- How noise affects edge detection
- Combining smoothing with edge detection (LoG filter)

**What you'll build:**
- Vertical and horizontal edge detectors
- A combined "super kernel" (Laplacian of Gaussian)

**Connection to previous labs:**
- Lab 7: You learned convolution mechanics
- Lab 8 (today): You'll use convolution to find edges
- Looking ahead: Edges are fundamental features for object recognition!

**Why edges matter:**
Edges represent significant changes in intensity — they mark object boundaries, surface orientations, and important structures. Most computer vision algorithms start by finding edges!

## Setup

In [ ]:
"""
Computer Vision Course - Lab 8: Discrete Derivatives

This cell sets up the environment.
Works automatically for both local and Google Colab!
"""

import os
import sys

# Detect environment
IN_COLAB = 'google.colab' in sys.modules

print("=" * 60)
print("Computer Vision - Lab 8: Edge Detection")
print("=" * 60)

if IN_COLAB:
    print("\n🔵 Running on Google Colab")
    print("-" * 60)
    
    if not os.path.exists('computer-vision'):
        print("📥 Cloning repository...")
        !git clone https://github.com/mjck/computer-vision.git
        print("✓ Repository cloned successfully")
    else:
        !git -C computer-vision pull
        print("✓ Repository updated")
    
    %cd computer-vision/labs/lab08_derivatives
    print(f"✓ Current directory: {os.getcwd()}")
    
    sys.path.insert(0, '/content/computer-vision')
    print("✓ Python path configured")
    
    print("-" * 60)
    print("🟢 Colab setup complete!\n")
    
else:
    print("\n🟢 Running locally")
    print("-" * 60)
    print(f"✓ Current directory: {os.getcwd()}")
    
    repo_root = os.path.abspath('../..')
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    print(f"✓ Repository root: {repo_root}")
    
    print("-" * 60)
    print("🟢 Local setup complete!\n")

print("=" * 60)
print("✅ Environment ready!")
print("=" * 60)

## Import Libraries

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy import signal

# Import course utilities
try:
    from sdx import cv_imread, cv_imshow, cv_grayread
    print("✓ sdx module loaded")
except ImportError as e:
    print(f"❌ Could not import sdx: {e}")
    raise

print("✓ All imports successful")

## Choosing an Image

Like Lab 7, we have **6 source images** available:
- **Photos:** `insper`, `informatica`, `harvard`
- **Logos:** `atletica`, `smash`, `consulting`

We'll start with clean images (no noise) to understand how edge detection works, then explore noise effects later.

In [ ]:
# Choose your source
SOURCE = 'insper'  # Options: 'insper', 'informatica', 'harvard', 'atletica', 'smash', 'consulting'

print(f"✓ Selected source: {SOURCE}")

### Load and Display Image

In [ ]:
# Load clean image (no noise)
input_image = cv_grayread(f'{SOURCE}.png', asfloat=True)

print(f"Image shape: {input_image.shape}")
print(f"Data type: {input_image.dtype}")

cv_imshow(input_image)
print(f"Showing: {SOURCE} (clean, no noise)")

## Convenience Functions

Now that you understand how convolution works (from Lab 7), we can use optimized library functions for speed.

### Convolution Function

We'll use SciPy's `correlate2d` instead of `convolve2d` because some edge detection kernels are **not symmetric**, and we don't want them to be flipped.

**Why `correlate2d`?**
- Convolution flips the kernel before applying it
- Correlation does not flip the kernel
- For symmetric kernels (like Gaussian), they're the same
- For asymmetric kernels (like Sobel), we want correlation

The `mode='valid'` parameter ignores borders, like your Lab 7 implementation.

In [ ]:
def convolve(input, kernel):
    """
    Apply 2D convolution using correlation (no kernel flip).
    
    Args:
        input: Input image
        kernel: Convolution kernel
    
    Returns:
        Convolved image (smaller due to border handling)
    """
    return signal.correlate2d(input, kernel, mode='valid')

print("✓ convolve() function defined")
print("  Using scipy.signal.correlate2d for efficiency")

### Normalization Function

Edge detection often produces values outside [0, 255]. This function normalizes outputs for display.

In [ ]:
def normalize(output):
    """
    Normalize an array to [0, 255] range.
    
    Maps:
        min(output) → 0
        max(output) → 255
    
    Args:
        output: Array to normalize
    
    Returns:
        Normalized array in [0, 255]
    """
    min_val = output.min()
    max_val = output.max()
    
    if max_val == min_val:
        return np.full_like(output, 127.5)
    
    return 255 * (output - min_val) / (max_val - min_val)

print("✓ normalize() function defined")
print("  Stretches values to full [0, 255] range for visualization")

---
## Sobel Filters

**The idea:** Edges are places where intensity changes rapidly. Mathematically, this means **high gradient magnitude**.

**Gradient** is a vector with two components:
- **∂I/∂x** — rate of change in x-direction (horizontal)
- **∂I/∂y** — rate of change in y-direction (vertical)

**Sobel filters** approximate these partial derivatives using convolution.

### Sobel Vertical Edge Filter

This kernel detects **vertical edges** (changes in the x-direction).

**Kernel Dx:**
```
[-1,  0,  1]
[-2,  0,  2]
[-1,  0,  1]
```

**How it works:**
- Negative weights on the left, positive on the right
- When a vertical edge passes through, left and right have different values
- This produces a strong response
- The middle row has weight 2 for stronger emphasis

In [ ]:
# Sobel kernel for vertical edges (gradient in x-direction)
Dx = np.array([
    [-1,  0,  1],
    [-2,  0,  2],
    [-1,  0,  1],
])

print("Dx kernel (vertical edge detector):")
print(Dx)

### Apply Vertical Edge Detection

Let's see what vertical edges look like:

In [ ]:
# Compute gradient in x-direction
gradient_x = convolve(input_image, Dx)

print("Vertical edge detection results:\n")

# Show absolute value (all edges bright regardless of direction)
cv_imshow(normalize(np.abs(gradient_x)))
print("Absolute value: All vertical edges shown as bright")

# Show raw gradient (negative = dark-to-light, positive = light-to-dark)
cv_imshow(normalize(gradient_x))
print("Raw gradient: Shows edge direction (brightness → darkness or vice versa)")

print(f"\nGradient range: [{gradient_x.min():.1f}, {gradient_x.max():.1f}]")

### Sobel Horizontal Edge Filter

This kernel detects **horizontal edges** (changes in the y-direction).

**Kernel Dy:**
```
[-1, -2, -1]
[ 0,  0,  0]
[ 1,  2,  1]
```

**How it works:**
- Negative weights on top, positive on bottom
- Detects horizontal edges
- Middle column has weight 2 for emphasis

In [ ]:
# Sobel kernel for horizontal edges (gradient in y-direction)
Dy = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1],
])

print("Dy kernel (horizontal edge detector):")
print(Dy)

### Apply Horizontal Edge Detection

In [ ]:
# Compute gradient in y-direction
gradient_y = convolve(input_image, Dy)

print("Horizontal edge detection results:\n")

cv_imshow(normalize(np.abs(gradient_y)))
print("Absolute value: All horizontal edges")

cv_imshow(normalize(gradient_y))
print("Raw gradient: Shows edge direction")

print(f"\nGradient range: [{gradient_y.min():.1f}, {gradient_y.max():.1f}]")

### Gradient Magnitude — All Edges

The gradient vector is `(∂I/∂x, ∂I/∂y)`. Its magnitude tells us how strong the edge is, regardless of direction.

**Formula:**
```
|∇I| = √((∂I/∂x)² + (∂I/∂y)²)
```

This combines vertical and horizontal edges into one image showing **all edges**.

In [ ]:
# Compute gradient magnitude
gradient_magnitude = np.sqrt(gradient_x**2 + gradient_y**2)

print("Combined gradient magnitude (all edges):")
cv_imshow(normalize(gradient_magnitude))

print(f"Magnitude range: [{gradient_magnitude.min():.1f}, {gradient_magnitude.max():.1f}]")
print("\n✓ Brighter pixels = stronger edges")

### Explore All Sources

Let's see how Sobel performs on all 6 images:

In [ ]:
print("Sobel edge detection on all sources:\n")

for source in ['insper', 'informatica', 'harvard', 'atletica', 'smash', 'consulting']:
    image = cv_grayread(f'{source}.png', asfloat=True)
    
    # Compute gradients
    gx = convolve(image, Dx)
    gy = convolve(image, Dy)
    magnitude = np.sqrt(gx**2 + gy**2)
    
    cv_imshow(normalize(magnitude))
    print(f"{source}")

print("\n🤔 Notice: Logos have cleaner edges than photos!")

---
## Laplacian Filter

The **Laplacian** is the sum of second partial derivatives:

**Formula:**
```
∇²I = ∂²I/∂x² + ∂²I/∂y²
```

**Kernel:**
```
[ 0,  1,  0]
[ 1, -4,  1]
[ 0,  1,  0]
```

**How it works:**
- Center pixel weighted by -4
- Four neighbors weighted by +1 each
- If all neighbors equal the center → Laplacian = 0
- If neighbors differ from center → Laplacian ≠ 0
- **Detects edges where curvature changes**

**Advantage over Sobel:** Single kernel instead of two, rotation-invariant

In [ ]:
# Laplacian kernel
L = np.array([
    [ 0,  1,  0],
    [ 1, -4,  1],
    [ 0,  1,  0],
])

print("Laplacian kernel:")
print(L)
print("\nInterpretation: Detects where center differs from neighbors")

### Apply Laplacian

In [ ]:
# Apply Laplacian filter
edges = convolve(input_image, L)

print("Laplacian edge detection:")
cv_imshow(normalize(np.abs(edges)))

print(f"Laplacian range: [{edges.min():.1f}, {edges.max():.1f}]")
print("\n✓ Detects all edges regardless of orientation")

### Explore All Sources with Laplacian

In [ ]:
print("Laplacian edge detection on all sources:\n")

for source in ['insper', 'informatica', 'harvard', 'atletica', 'smash', 'consulting']:
    image = cv_grayread(f'{source}.png', asfloat=True)
    edges = convolve(image, L)
    cv_imshow(normalize(np.abs(edges)))
    print(f"{source}")

print("\n🤔 Compare to Sobel — which works better?")

---
## The Problem of Noise

Edge detectors respond to **rapid changes in intensity**. Unfortunately, **noise also creates rapid changes**!

Let's see what happens when we apply edge detection to noisy images.

### Select Noise Severity

Same as Lab 7:
- `8` — Light noise
- `16` — Medium noise
- `32` — Heavy noise

In [ ]:
# Choose noise level
SEVERITY = 8  # Options: 8, 16, 32

print(f"✓ Selected severity: {SEVERITY}%")

### Load Noisy Image

In [ ]:
# Load noisy version
noisy_input = cv_grayread(f'{SOURCE}-{SEVERITY}.png', asfloat=True)

print("Noisy image:")
cv_imshow(noisy_input)
print(f"{SOURCE} with {SEVERITY}% noise")

### Edge Detection on Noisy Image

Watch what happens when we apply Laplacian to the noisy image:

In [ ]:
# Apply Laplacian to noisy image
noisy_edges = convolve(noisy_input, L)

print("Edge detection on noisy image:")
cv_imshow(normalize(np.abs(noisy_edges)))

print("\n⚠️ Problem: Noise creates false edges everywhere!")
print("The real edges are buried in noise responses.")

### 🤔 The Dilemma

**The problem:** We want edges but not noise responses.

**Two conflicting goals:**
1. **Smooth** the image to reduce noise
2. **Detect edges** which are high-frequency features

**The insight:** What if we do both at once?

---
## Exercise: Two-Step Solution

**Idea:** First smooth the image, *then* detect edges.

**Your task:** Apply Gaussian smoothing followed by Laplacian edge detection.

**Steps:**
1. Create a Gaussian kernel (use what you learned in Lab 7!)
2. Smooth the noisy image
3. Apply Laplacian to the smoothed image

**Hint:** You can reuse your `gaussian_kernel()` from Lab 7, or create one directly.

In [ ]:
# ── Your code here ──────────────────────────────────────────────────────────
# Step 1: Create a Gaussian kernel
# Hint: You can use the formula from Lab 7, or define it directly

# Step 2: Smooth the noisy image
# smoothed = ...

# Step 3: Apply Laplacian to smoothed image
# edges_from_smoothed = ...

# Step 4: Display the result
# cv_imshow(normalize(np.abs(edges_from_smoothed)))

# ────────────────────────────────────────────────────────────────────────────

print("Implement the two-step process above")

### Expected Result

If done correctly, you should see:
- ✅ True edges clearly visible
- ✅ Much less noise response
- ⚠️ Some edge details lost (trade-off!)

---
## Challenge: The Laplacian of Gaussian (LoG)

**Key insight from convolution theory:**

If you convolve with kernel A, then convolve the result with kernel B:
```
(Image ⊗ A) ⊗ B = Image ⊗ (A ⊗ B)
```

This means **we can combine the two operations into one super kernel!**

**Your task:** Create a single kernel that performs both Gaussian smoothing AND Laplacian edge detection in one convolution.

**How:**
1. Create a Gaussian kernel G
2. Create a Laplacian kernel L
3. Convolve them together: `K = convolve(G, L)`
4. Use K on the noisy image

This combined kernel is called the **Laplacian of Gaussian (LoG)** or **Mexican Hat** filter.

In [ ]:
# ── Your code here ──────────────────────────────────────────────────────────
# Create the combined kernel K that does both operations
# Hint: convolve the Gaussian and Laplacian kernels together

K = np.array([
    [0, 0, 0],
    [0, 1, 0],
    [0, 0, 0],
])  # Replace this identity kernel with your solution

# ────────────────────────────────────────────────────────────────────────────

print("LoG kernel (Laplacian of Gaussian):")
print(K)
print(f"Kernel shape: {K.shape}")

### Test Your Super Kernel

If your kernel is correct, applying it once should give the same result as smoothing then detecting edges!

In [ ]:
# Apply your combined kernel
output = convolve(noisy_input, K)

print("Single-kernel result:")
cv_imshow(normalize(np.abs(output)))

print("\n✓ This should match your two-step result!")
print("✓ But it's done with a SINGLE convolution")

### Explore Different Noise Levels

In [ ]:
print(f"LoG filter on {SOURCE} with varying noise:\n")

for severity in [8, 16, 32]:
    image = cv_grayread(f'{SOURCE}-{severity}.png', asfloat=True)
    result = convolve(image, K)
    cv_imshow(normalize(np.abs(result)))
    print(f"{severity}% noise")

print("\n🤔 How does it perform as noise increases?")

### Explore Different Sources

In [ ]:
print(f"LoG filter on all sources ({SEVERITY}% noise):\n")

for source in ['insper', 'informatica', 'harvard', 'atletica', 'smash', 'consulting']:
    image = cv_grayread(f'{source}-{SEVERITY}.png', asfloat=True)
    result = convolve(image, K)
    cv_imshow(normalize(np.abs(result)))
    print(f"{source}")

print("\n✓ LoG filter working across all images!")

---
## OpenCV's Built-in Edge Detectors

Now that you understand the principles, here are OpenCV's optimized implementations:

**Documentation:**
- [cv2.Sobel](https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html#gacea54f142e81b6758cb6f375ce782c8d) — Sobel derivative
- [cv2.Laplacian](https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html#gad78703e4c8fe703d479c1860d76429e6) — Laplacian edge detector
- [cv2.Canny](https://docs.opencv.org/4.x/dd/d1a/group__imgproc__feature.html#ga04723e007ed888ddf11d9ba04e2232de) — Canny edge detector (more sophisticated)

**Example:**

In [ ]:
# OpenCV Sobel
sobel_x = cv2.Sobel(input_image, cv2.CV_32F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(input_image, cv2.CV_32F, 0, 1, ksize=3)
sobel_mag = np.sqrt(sobel_x**2 + sobel_y**2)

print("OpenCV Sobel edge detection:")
cv_imshow(normalize(sobel_mag))

# OpenCV Laplacian
laplacian = cv2.Laplacian(input_image, cv2.CV_32F)

print("\nOpenCV Laplacian edge detection:")
cv_imshow(normalize(np.abs(laplacian)))

print("\n✓ Compare to your implementations!")

---
## Summary

### What You Learned

1. **Edge detection fundamentals**
   - Edges = rapid intensity changes
   - Gradients measure these changes
   - Derivatives can be computed via convolution

2. **Sobel filters**
   - Dx detects vertical edges (x-gradient)
   - Dy detects horizontal edges (y-gradient)
   - Magnitude combines both

3. **Laplacian filter**
   - Second derivative operator
   - Rotation-invariant
   - Single kernel for all orientations

4. **The noise problem**
   - Edge detectors respond to noise
   - Need to balance smoothing vs edge preservation

5. **Laplacian of Gaussian (LoG)**
   - Smooth first, then detect
   - Can be combined into one kernel!
   - Fundamental insight: convolution is associative

### Key Insights

**Why edge detection matters:**
- Edges mark object boundaries
- Reduce data (from millions of pixels to thousands of edges)
- Foundation for higher-level vision
- Used in: OCR, face detection, medical imaging, autonomous vehicles

**The noise-edges dilemma:**
- Can't have perfect edges AND no noise
- Must choose: more smoothing (lose details) or less smoothing (keep noise)
- LoG gives good balance

**Mathematical elegance:**
- (Image ⊗ G) ⊗ L = Image ⊗ (G ⊗ L)
- Two operations combined into one!
- This principle extends to CNNs

### Connection to Deep Learning

**In modern CNNs:**
- Early layers learn edge-like filters automatically
- Often look similar to Sobel/Laplacian
- But learned from data, not hand-designed
- Deeper layers build on these edge features

**What you built today:**
- The same filters CNNs discover on their own
- Understanding these helps interpret what CNNs learn

### ✏️ Final Reflection

Answer these questions:

1. Why do edge detectors respond strongly to noise?
2. What's the advantage of LoG over applying Gaussian and Laplacian separately?
3. Why might Sobel be preferred over Laplacian in some cases?
4. How does edge detection connect to object recognition?

In [ ]:
# Your reflection:

# 1. Why do edge detectors respond to noise?
#    ...

# 2. Advantage of LoG (single kernel)?
#    ...

# 3. When prefer Sobel over Laplacian?
#    ...

# 4. Connection to object recognition?
#    ...


---
## 📋 Submission Checklist

Before submitting, make sure:

- [ ] Understood Sobel vertical and horizontal filters
- [ ] Examined gradient magnitude on all sources
- [ ] Understood Laplacian filter
- [ ] Saw the noise problem with edge detection
- [ ] Exercise: Implemented smooth-then-detect (Gaussian + Laplacian)
- [ ] Challenge: Created combined LoG kernel
- [ ] Tested LoG on different noise levels and sources
- [ ] Final reflection questions answered
- [ ] All cells executed in order

**Bonus (+1 point):** Test Canny edge detector on sample images and compare to Laplacian or Sobel results.